## Smoothing of Athens AMS PM0.1 size distribution data

In [1]:
#Install packages
%pip install pandas matplotlib scipy openpyxl python-pptx

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\GeorgiaRg\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
#Install libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches
from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline

In [3]:
#Define working directories
wdir = Path("C:/Users/GeorgiaRg/Documents/Athens2025/AMS/ForCE/python_smoothing")
os.chdir(wdir) # Change directory
output_dir = r"C:\Users\GeorgiaRg\Documents\Athens2025\AMS\ForCE\python_smoothing\output" # Define the name or path of your output folder
os.makedirs(output_dir, exist_ok=True) # exist_ok=True prevents an error if you run this cell multiple times

In [4]:
#Import data
df_org = pd.read_excel(r"C:/Users/GeorgiaRg/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="Org")
df_sulf = pd.read_excel(r"C:/Users/GeorgiaRg/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="Sulf")
df_amm = pd.read_excel(r"C:/Users/GeorgiaRg/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="Amm")
df_nitr = pd.read_excel(r"C:/Users/GeorgiaRg/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="Nitr")
df_dva = pd.read_excel(r"C:/Users/GeorgiaRg/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="diameter")

In [ ]:
df_org.head()

In [5]:
# 1. Group your dataframes into a dictionary so we can loop through them easily
dfs = {
    'Org': df_org,
    'Sulf': df_sulf,
    'Amm': df_amm,
    'Nitr': df_nitr,
}

# Define your cutoff date and time
cutoff_date = pd.to_datetime('2025-01-24 00:00:00')

# 2. Loop through and clean each dataframe
# 3. Clean and filter (Bulletproof version)
for name, df in dfs.items():

    # Grab the exact name of the first column (index 0) and rename it
    first_col_name = df.columns[0]
    df.rename(columns={first_col_name: 'Datetime'}, inplace=True)

    # Convert to datetime
    df['Datetime'] = pd.to_datetime(df['Datetime'])

    # Filter out anything before Jan 24, 2025
    df_filtered = df[df['Datetime'] >= cutoff_date]

    # Set index
    df_filtered.set_index('Datetime', inplace=True)

    # Update dictionary
    dfs[name] = df_filtered

# 3. Reassign the cleaned dataframes back to your original variables
df_org = dfs['Org']
df_sulf = dfs['Sulf']
df_amm = dfs['Amm']
df_nitr = dfs['Nitr']

# Check your work
print(f"df_org now starts on: {df_org.index.min()}")

df_org now starts on: 2025-01-24 00:00:01.508000


In [6]:
#Ammonium for PM0.1 Athens was not doing so good. Replace it here with Sulfate times 2 here if necessary
df_amm = df_sulf * 0.375 #NH4 = (18*sulf*2)/96. 1 mol of sulf is equal to 2 moles of ammonium, assuming that all NH3 is neutralized as ammonium sulfate

In [7]:
df_amm

,Dp1,Dp2,Dp3,Dp4,Dp5,Dp6,Dp7,Dp8,Dp9,Dp10,...,Dp245,Dp246,Dp247,Dp248,Dp249,Dp250,Dp251,Dp252,Dp253,Dp254
Datetime,,,,,,,,,,,,,,,,,,,,,
2025-01-24 00:00:01.508,-0.023586,0.014274,0.014999,-0.017218,-0.022783,0.005540,-0.007478,0.010226,-0.007981,-0.007142,...,0.078411,-0.007153,-0.008596,0.027529,-0.038224,-0.067084,0.014362,0.037028,-0.000623,0.012905
2025-01-24 00:03:00.824,-0.033478,0.020928,0.000282,-0.004371,-0.027475,0.008208,-0.008424,-0.006691,-0.013288,-0.020812,...,0.058146,-0.069961,0.034928,-0.069631,-0.031012,0.021214,0.024344,-0.002239,-0.041050,-0.029119
2025-01-24 00:06:01.195,-0.046999,0.008594,-0.018513,0.025138,-0.017785,0.013039,0.004728,-0.006853,-0.006291,-0.001772,...,0.050870,0.056478,0.015535,-0.012847,-0.051604,-0.003913,-0.016959,0.036486,0.033403,0.023357
2025-01-24 00:09:02.493,0.004583,-0.028403,0.000166,-0.008084,0.021477,0.008748,-0.018097,0.009465,-0.003296,-0.006900,...,0.062091,-0.031763,0.035360,-0.005519,-0.076412,-0.017512,0.003217,-0.041494,-0.035974,-0.001190
2025-01-24 00:12:03.549,0.009511,-0.007774,0.002547,0.010859,0.008804,0.025778,-0.004377,-0.004165,-0.002515,-0.015934,...,-0.018558,-0.003715,-0.008758,0.045597,-0.023830,0.009014,0.029114,-0.045310,0.036633,0.030676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-21 09:21:07.773,0.026151,-0.003005,-0.013214,-0.042859,-0.017041,0.018049,-0.015464,0.019387,0.004242,0.003224,...,0.036562,0.063261,-0.013792,-0.002716,-0.042215,-0.046438,-0.011700,-0.043542,0.028165,-0.019223
2025-02-21 09:24:09.770,-0.004958,0.008376,0.020980,-0.032020,-0.009257,0.003801,-0.001963,0.006427,0.016411,-0.008938,...,-0.051684,-0.035604,0.020269,-0.011975,0.037624,0.017333,0.009850,0.011271,0.030135,-0.006097
2025-02-21 09:27:14.883,-0.003798,-0.016669,-0.012164,-0.048679,-0.012116,-0.004322,-0.013878,0.002418,0.004730,0.006030,...,0.035940,-0.023134,-0.003765,0.090183,-0.034217,0.013784,-0.056556,0.062433,-0.042698,0.006808


In [8]:
df_sulf

,Dp1,Dp2,Dp3,Dp4,Dp5,Dp6,Dp7,Dp8,Dp9,Dp10,...,Dp245,Dp246,Dp247,Dp248,Dp249,Dp250,Dp251,Dp252,Dp253,Dp254
Datetime,,,,,,,,,,,,,,,,,,,,,
2025-01-24 00:00:01.508,-0.062895,0.038064,0.039998,-0.045913,-0.060754,0.014773,-0.019942,0.027270,-0.021283,-0.019045,...,0.209095,-0.019075,-0.022924,0.073410,-0.101930,-0.178892,0.038300,0.098741,-0.001662,0.034413
2025-01-24 00:03:00.824,-0.089275,0.055807,0.000753,-0.011655,-0.073266,0.021888,-0.022465,-0.017842,-0.035434,-0.055500,...,0.155055,-0.186562,0.093141,-0.185684,-0.082698,0.056572,0.064919,-0.005969,-0.109466,-0.077652
2025-01-24 00:06:01.195,-0.125331,0.022918,-0.049367,0.067034,-0.047427,0.034770,0.012609,-0.018274,-0.016777,-0.004725,...,0.135652,0.150609,0.041428,-0.034259,-0.137610,-0.010435,-0.045225,0.097296,0.089075,0.062286
2025-01-24 00:09:02.493,0.012222,-0.075741,0.000441,-0.021558,0.057272,0.023328,-0.048258,0.025240,-0.008789,-0.018400,...,0.165576,-0.084701,0.094294,-0.014717,-0.203765,-0.046699,0.008580,-0.110651,-0.095930,-0.003174
2025-01-24 00:12:03.549,0.025364,-0.020730,0.006791,0.028957,0.023477,0.068740,-0.011673,-0.011108,-0.006708,-0.042491,...,-0.049489,-0.009907,-0.023355,0.121591,-0.063546,0.024037,0.077636,-0.120828,0.097687,0.081803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-21 09:21:07.773,0.069737,-0.008013,-0.035236,-0.114292,-0.045443,0.048131,-0.041238,0.051698,0.011312,0.008596,...,0.097497,0.168696,-0.036777,-0.007242,-0.112572,-0.123834,-0.031199,-0.116113,0.075107,-0.051261
2025-02-21 09:24:09.770,-0.013222,0.022335,0.055947,-0.085387,-0.024684,0.010136,-0.005236,0.017140,0.043763,-0.023835,...,-0.137825,-0.094944,0.054052,-0.031934,0.100331,0.046221,0.026266,0.030057,0.080361,-0.016259
2025-02-21 09:27:14.883,-0.010127,-0.044451,-0.032437,-0.129812,-0.032310,-0.011526,-0.037008,0.006449,0.012613,0.016079,...,0.095841,-0.061690,-0.010041,0.240488,-0.091245,0.036759,-0.150816,0.166489,-0.113861,0.018155


In [ ]:
df_org.head()

In [ ]:
# Create new dataframes for all the species that do have the negative values replaced with zero, setting a "floor" of 0 for all numerical values
# The .clip() method automatically returns a new DataFrame rather than modifying the old one

#df_org_pos = df_org.clip(lower=0)
#df_sulf_pos = df_sulf.clip(lower=0)
#df_amm_pos = df_amm.clip(lower=0)
#df_nitr_pos = df_nitr.clip(lower=0)

# Quick check to verify it worked
#print(f"Minimum value in old Org dataframe: {df_org.min().min()}")
#print(f"Minimum value in new Org dataframe: {df_org_pos.min().min()}")

In [ ]:
# 1. Ensure we have our 1D list of physical diameters
#diameters = df_dva.values.flatten()

# 2. Identify which columns correspond to diameters > 500 nm
# This creates a list of the specific column names (e.g., 'Dp150', 'Dp151'...)
#cols_to_zero = df_org_pos.columns[diameters > 300]

# 3. Force all values in those specific columns to exactly 0 for all positive dataframes
#df_org_pos.loc[:, cols_to_zero] = 0
#df_sulf_pos.loc[:, cols_to_zero] = 0
#df_amm_pos.loc[:, cols_to_zero] = 0
#df_nitr_pos.loc[:, cols_to_zero] = 0
# 3. Force all values in those specific columns to exactly 0 for all negative dataframes
#df_org.loc[:, cols_to_zero] = 0
#df_sulf.loc[:, cols_to_zero] = 0
#df_amm.loc[:, cols_to_zero] = 0
#df_nitr.loc[:, cols_to_zero] = 0

# Quick check to verify how many bins were zeroed out
#print(f"Total number of size bins: {len(diameters)}")
#print(f"Number of size bins > 300nm forced to zero: {len(cols_to_zero)}")

In [8]:
# ==========================================
# 1. SETUP PARAMETERS & DIAMETERS
# ==========================================
window_size = 11
poly_order = 3

# Dilution correction factor (Total Flow / Sample Flow)
dilution_factor = 4 / 1.5

# Extract the physical diameters from df_dva by flattening its values
diameters = df_dva.values.flatten()

# Quick safety check to ensure sizes match before plotting
if len(diameters) != len(df_org.columns):
    raise ValueError(f"Mismatch! Found {len(diameters)} diameters but {len(df_org.columns)} size bins.")

# Group the dataframes
species_dfs = {
    'Org': df_org,
    'SO4': df_sulf,
    'NH4': df_amm,
    'NO3': df_nitr
}

averaged_dfs = {}

# ==========================================
# 2. APPLY SMOOTHING, DILUTION, & AVERAGING
# ==========================================
print("Smoothing data, applying dilution correction, and averaging to 4-hour intervals...")

for species, df in species_dfs.items():
    # Replace infinities with NaNs, then fill all NaNs with 0
    df_clean = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    # 1. Apply Savitzky-Golay filter across the columns (keeping all negatives!)
    smoothed_array = savgol_filter(df_clean.values, window_length=window_size, polyorder=poly_order, axis=1)
    df_smoothed = pd.DataFrame(smoothed_array, index=df_clean.index, columns=df_clean.columns)

    # 2. Apply the dilution correction
    df_smoothed = df_smoothed * dilution_factor

    # 3. Resample to 4-hour averages (Let the positive and negative noise cancel out here!)
    df_4h_avg = df_smoothed.resample('4h').mean()

    # 4. NOW clip the final averaged data to 0
    averaged_dfs[species] = df_4h_avg.clip(lower=0)

# ==========================================
# 3. GENERATE POWERPOINT PRESENTATION
# ==========================================
print("Generating PowerPoint slides...")

prs = Presentation()
prs.slide_width = Inches(13.333) # 16:9 widescreen
prs.slide_height = Inches(7.5)

timestamps = averaged_dfs['Org'].index

for timestamp in timestamps:
    # Skip plotting if this specific hour chunk is completely empty
    if averaged_dfs['Org'].loc[timestamp].isna().all():
        continue

    fig, ax = plt.subplots(figsize=(12, 7))

    # Plot each species
    ax.plot(diameters, averaged_dfs['NH4'].loc[timestamp], label='NH4', color='#FFA500', linewidth=2.5)
    ax.plot(diameters, averaged_dfs['Org'].loc[timestamp], label='Org', color='#228B22', linewidth=2.5)
    ax.plot(diameters, averaged_dfs['SO4'].loc[timestamp], label='SO4', color='#FF2400', linewidth=2.5)
    ax.plot(diameters, averaged_dfs['NO3'].loc[timestamp], label='NO3', color='#87CEEB', linewidth=2.5)

    # Formatting X-axis (Log scale for particle diameters)
    ax.set_xscale('log')
    ax.set_xlim([10, 2000])

    # Formatting Labels
    ax.set_xlabel('Particle Diameter (nm)', fontsize=14)
    ax.set_ylabel('dM/dlogDva (µg m⁻³)', fontsize=14)

    # Formatting Title
    formatted_time = timestamp.strftime('%Y-%m-%d %H:%M')
    ax.set_title(f'AMS size distributions (CE=1) – {formatted_time}', fontsize=16, loc='left', pad=40)

    # Formatting Legend
    ax.legend(title='Species', loc='upper center', bbox_to_anchor=(0.5, 1.10),
              ncol=4, frameon=False, fontsize=12, title_fontsize=14)

    # Formatting Grid
    ax.grid(True, which="major", ls="-", alpha=0.3)
    ax.grid(True, which="minor", ls="-", alpha=0.1)

    # Save and insert image
    temp_img = "temp_slide_plot.png"
    fig.savefig(temp_img, bbox_inches='tight', dpi=150)
    plt.close(fig)

    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide.shapes.add_picture(temp_img, Inches(0.5), Inches(0.5), width=Inches(10))

# ==========================================
# 4. SAVE TO SPECIFIC DIRECTORY
# ==========================================
output_dir = r"C:\Users\GeorgiaRg\Documents\Athens2025\AMS\ForCE\python_smoothing\output"
os.makedirs(output_dir, exist_ok=True)

output_filepath = os.path.join(output_dir, "AMS_Smoothed_4Hour_Averages_neg_DilutionCorrected_Amm_2xSulf.pptx")

# Save the presentation directly to that path
prs.save(output_filepath)

# Clean up the temporary image
if os.path.exists("temp_slide_plot.png"):
    os.remove("temp_slide_plot.png")

print(f"The PowerPoint is saved here:\n{output_filepath}")

Smoothing data, applying dilution correction, and averaging to 4-hour intervals...
Generating PowerPoint slides...
The PowerPoint is saved here:
C:\Users\GeorgiaRg\Documents\Athens2025\AMS\ForCE\python_smoothing\output\AMS_Smoothed_4Hour_Averages_neg_DilutionCorrected_Amm_2xSulf.pptx


In [9]:
# ==========================================
# 5. EXPORT DATA TO EXCEL
# ==========================================
print("Exporting data to Excel...")
excel_filepath = os.path.join(output_dir, "AMS_Smoothed_4Hour_Averages_neg_DilutionCorrected_Amm_2xSulf.xlsx")

# Use pandas ExcelWriter to create multiple sheets in one file
with pd.ExcelWriter(excel_filepath) as writer:
    # Save the physical diameters to the first sheet so you have the X-axis reference
    pd.DataFrame({'Diameter_nm': diameters}).to_excel(writer, sheet_name='Diameters', index=False)

    # Loop through the smoothed & averaged dataframes and save each to its own sheet
    for species, df in averaged_dfs.items():
        df.to_excel(writer, sheet_name=species)


print(f"The Excel data is saved here:\n{excel_filepath}")

Exporting data to Excel...
The Excel data is saved here:
C:\Users\GeorgiaRg\Documents\Athens2025\AMS\ForCE\python_smoothing\output\AMS_Smoothed_4Hour_Averages_neg_DilutionCorrected_Amm_2xSulf.xlsx
